# PD Knowledge Base feature-matching baseline validation

This notebook expands the original feature-matching demo into a 75-case validation and calibration harness. It evaluates the existing deterministic and RapidFuzz/TF-IDF matcher without changing its algorithm, weights, terminology, Knowledge Base, or acceptance behavior. No LLM is invoked.

## Ground-truth isolation

Only `feature_name` and `description` cross the matcher boundary. Expected feature, match type, representation type, and category remain in a separate ground-truth frame and are joined to predictions only after every matcher call completes.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from functions.feature_matching import load_kb, load_terminology
from functions.feature_matching_validation import (
    compact_evaluation_output,
    compare_validation_outputs,
    validate_feature_matcher,
)

KB_PATH = HERE / 'kb' / 'pd_directionality_kb_v0_3.yaml'
TERMINOLOGY_PATH = HERE / 'kb' / 'credit_risk_abbreviations_v0_2.yaml'
VALIDATION_PATH = HERE / 'inputs' / 'feature_matching_validation_v0_1.csv'
CSV_OUTPUT_PATH = HERE / 'output' / 'feature_matching_validation_results_v0_1_kb_v0_3.csv'
previous_results = (
    pd.read_csv(CSV_OUTPUT_PATH, keep_default_na=False)
    if CSV_OUTPUT_PATH.exists()
    else None
)

kb = load_kb(KB_PATH)
terminology = load_terminology(TERMINOLOGY_PATH)

In [2]:
artifacts = validate_feature_matcher(
    VALIDATION_PATH,
    kb,
    terminology,
    top_n=3,
)

validation = artifacts['validation']
evaluation = artifacts['evaluation']
errors = artifacts['errors']

print(f'Loaded {len(validation)} labelled cases.')
display(validation.groupby(['test_category', 'expected_match_type']).size().rename('cases').reset_index())

Loaded 75 labelled cases.


,test_category,expected_match_type,cases
0,abbreviated,match,15
1,ambiguous,match,4
2,ambiguous,no_match,1
3,description_assisted,match,15
4,easy,match,10
5,inverse_representation,match,5
6,out_of_kb,no_match,10
7,related_concepts,match,15


## Overall baseline

Rates are proportions. Deterministic exact matches and NLP candidates are treated consistently for Top-1/Top-3 retrieval. An exact match contributes its one canonical feature to Top-3.

In [3]:
display(artifacts['overall_summary'])

,metric,count,rate
0,validation_cases,75,NaN
1,matchable_cases,64,NaN
2,top1_correct,62,0.968750
3,top3_correct,64,1.000000
4,deterministic_exact_matches,18,NaN
5,deterministic_exact_correct,18,1.000000
6,expected_no_match_cases,11,NaN
7,no_match_correct,4,0.363636
8,false_match_count,7,0.636364
9,inverse_cases,6,NaN


## Performance by test category

In [4]:
display(artifacts['category_summary'])

,test_category,cases,matchable_cases,top1_correct,top1_accuracy,top3_correct,top3_accuracy,expected_no_match_cases,no_match_correct,no_match_accuracy
0,abbreviated,15,15,14,0.933333,15,1.0,0,0,NaN
1,ambiguous,5,4,4,1.000000,4,1.0,1,0,0.0
2,description_assisted,15,15,15,1.000000,15,1.0,0,0,NaN
3,easy,10,10,10,1.000000,10,1.0,0,0,NaN
4,inverse_representation,5,5,5,1.000000,5,1.0,0,0,NaN
5,out_of_kb,10,0,0,NaN,0,NaN,10,4,0.4
6,related_concepts,15,15,14,0.933333,15,1.0,0,0,NaN


## Incorrect Top-1 predictions

In [5]:
display(errors['incorrect_top1'])

,feature_name,description,expected_canonical_feature,top_candidate,combined_score,second_candidate,second_score,score_gap,test_category
0,DTI_CURR,Current total monthly debt obligations divided...,debt_service_burden,income_capacity,0.49648,debt_service_burden,0.471179,0.025301,abbreviated
1,MONTHLY_DEBT_INC,Monthly debt obligations divided by monthly bo...,debt_service_burden,income_capacity,0.59009,debt_service_burden,0.500949,0.089141,related_concepts


## Top-1 wrong but expected concept recovered in Top-3

These are candidate sets that a future adjudication layer could potentially resolve.

In [6]:
display(errors['top1_wrong_top3_correct'])

,feature_name,description,expected_canonical_feature,top_candidate,combined_score,second_candidate,second_score,score_gap,test_category,top_3_candidates
0,DTI_CURR,Current total monthly debt obligations divided...,debt_service_burden,income_capacity,0.49648,debt_service_burden,0.471179,0.025301,abbreviated,"[{'canonical_feature': 'income_capacity', 'fea..."
1,MONTHLY_DEBT_INC,Monthly debt obligations divided by monthly bo...,debt_service_burden,income_capacity,0.59009,debt_service_burden,0.500949,0.089141,related_concepts,"[{'canonical_feature': 'income_capacity', 'fea..."


## Expected concepts missing from Top-3

These are candidate-generation failures.

In [7]:
display(errors['expected_missing_top3'])

,feature_name,description,expected_canonical_feature,top_candidate,combined_score,second_candidate,second_score,score_gap,test_category,top_3_candidates


## Expected no-match cases returned as candidate matches

Low-scoring candidates retained under an explicit `no_match` status are not included here.

In [8]:
display(errors['false_matches'])

,feature_name,description,match_status,top_candidate,combined_score,second_candidate,second_score,score_gap
0,RR_METRIC,Recovery rate on previously defaulted exposures,candidate_match,cash_flow_growth,0.248349,noi_growth,0.241948,0.006401
1,BORROWER_ZIP_CODE,Borrower's postal or ZIP code,candidate_match,income_growth,0.308017,income_capacity,0.299085,0.008932
2,PROPERTY_TYPE,"Property category such as office, retail, indu...",candidate_match,property_value_growth,0.287389,cre_price_growth,0.281923,0.005466
3,LOAN_PURPOSE,"Purpose of borrowing such as purchase, refinan...",candidate_match,loan_to_cost,0.284228,debt_yield,0.267168,0.017060
4,INDUSTRY_CODE,Industry classification code for the obligor,candidate_match,credit_history_length,0.268649,cash_flow_growth,0.217054,0.051595
5,CURRENCY_CODE,Currency denomination of the facility,candidate_match,utilization,0.281962,unemployment_rate_change,0.212389,0.069573
6,ORIGINATION_CHANNEL,Channel through which the loan was originated,candidate_match,loan_to_value,0.321704,debt_yield,0.255229,0.066475


## Wrong deterministic exact matches and inverse-representation errors

In [9]:
print('Wrong deterministic exact matches')
display(errors['wrong_deterministic_exact'])

print('Incorrect inverse identifications')
display(errors['incorrect_inverse'])

Wrong deterministic exact matches


,feature_name,description,expected_canonical_feature,top_candidate,combined_score,second_candidate,second_score,score_gap,test_category,match_source_type


Incorrect inverse identifications


,feature_name,description,expected_canonical_feature,top_candidate,expected_representation_type,predicted_representation_type,match_source_type,combined_score
0,RENT_UNCOLLECTED_PCT,Percentage of rent due that was not collected,rent_collection,rent_collection,inverse,same_orientation,representation,0.600691
1,RR_SCORE,Internal borrower risk rating where larger val...,credit_quality_score,credit_quality_score,inverse,same_orientation,canonical,0.431101


## Score distributions

The table describes Top-1 combined score and Top-1-minus-Top-2 gap for correct matches, incorrect matches, and expected no-match cases. Exact matches have no score gap. This is descriptive evidence only; no thresholds are selected here.

In [10]:
display(artifacts['score_distributions'])

,group,metric,count,minimum,p25,median,p75,maximum,mean
0,correct_top1,combined_score,62,0.323326,0.489636,0.593814,1.000000,1.000000,0.671417
1,correct_top1,score_gap,44,0.064325,0.114529,0.158788,0.244115,0.363412,0.179525
2,incorrect_top1,combined_score,2,0.496480,0.519883,0.543285,0.566688,0.590090,0.543285
3,incorrect_top1,score_gap,2,0.025301,0.041261,0.057221,0.073181,0.089141,0.057221
4,expected_no_match,combined_score,11,0.128713,0.219826,0.268649,0.285809,0.321704,0.248962
5,expected_no_match,score_gap,11,0.001253,0.005933,0.010270,0.034327,0.069573,0.022829


## CSV output

Compare the new compact evaluation with the previously generated CSV, display any deviations, and then write the current 75-row result. The large nested matcher dictionaries and full candidate dictionaries remain available in memory but are intentionally omitted from the flat file.

In [11]:
csv_output = compact_evaluation_output(evaluation)
deviations = (
    compare_validation_outputs(previous_results, csv_output)
    if previous_results is not None
    else None
)
if deviations is None:
    print('No previous CSV was available for regression comparison.')
elif deviations.empty:
    print('Regression comparison: no deviations detected.')
else:
    print(f'Regression comparison: {len(deviations)} deviation(s) detected.')
    display(deviations)

csv_output.to_csv(CSV_OUTPUT_PATH, index=False)
print(f'Wrote {len(csv_output)} rows to {CSV_OUTPUT_PATH}')

Regression comparison: no deviations detected.
Wrote 75 rows to C:\Src\DataWorkbench\experiments\test-lab\t2_d11_dir_consistency\output\feature_matching_validation_results_v0_1.csv


# Sentence-embedding challenger

Controlled comparison of deterministic exact matching plus `sentence-transformers/all-MiniLM-L6-v2` against the unchanged RapidFuzz/TF-IDF baseline. Semantic ranking uses cosine similarity only and applies no rejection threshold.

In [12]:
from functions.semantic_feature_matching import build_semantic_index, load_sentence_transformer
from functions.semantic_feature_matching_validation import validate_semantic_challenger

SEMANTIC_CSV_PATH = HERE / 'output' / 'feature_matching_semantic_challenger_results_v0_1_kb_v0_3.csv'
semantic_model = load_sentence_transformer()
semantic_index = build_semantic_index(kb, terminology, model=semantic_model)
semantic_artifacts = validate_semantic_challenger(
    str(VALIDATION_PATH),
    str(CSV_OUTPUT_PATH),
    kb,
    terminology,
    semantic_index,
)
semantic_evaluation = semantic_artifacts['evaluation']
semantic_evaluation.to_csv(SEMANTIC_CSV_PATH, index=False)
print(f'Model: {semantic_index.model_name}')
print(f'Sentence Transformers: {semantic_index.package_version}')
print(f'Embedding dimension: {semantic_index.embedding_dimension}')
print(f'Wrote {len(semantic_evaluation)} rows to {SEMANTIC_CSV_PATH}')

C:\Src\DataWorkbench\experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1432.14it/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Sentence Transformers: 5.7.0
Embedding dimension: 384
Wrote 75 rows to C:\Src\DataWorkbench\experiments\test-lab\t2_d11_dir_consistency\output\feature_matching_semantic_challenger_results_v0_1.csv


## Challenger metrics and category performance

In [13]:
display(semantic_artifacts['challenger_summary'])
display(semantic_artifacts['category_summary'])

,metric,correct,cases,accuracy
0,top1,54,64,0.843750
1,top3,62,64,0.968750
2,embedding_only_top1,53,64,0.828125
3,embedding_only_top3,61,64,0.953125
4,exact,18,18,1.000000
5,no_match,0,11,0.000000
6,inverse,4,6,0.666667


,test_category,cases,matchable_cases,top1_correct,top1_accuracy,top3_correct,top3_accuracy,expected_no_match_cases,no_match_correct,no_match_accuracy
0,abbreviated,15,15,12,0.800000,14,0.933333,0,0,NaN
1,ambiguous,5,4,3,0.750000,4,1.000000,1,0,0.0
2,description_assisted,15,15,13,0.866667,15,1.000000,0,0,NaN
3,easy,10,10,10,1.000000,10,1.000000,0,0,NaN
4,inverse_representation,5,5,5,1.000000,5,1.000000,0,0,NaN
5,out_of_kb,10,0,0,NaN,0,NaN,10,0,0.0
6,related_concepts,15,15,11,0.733333,14,0.933333,0,0,NaN


## Baseline versus challenger and all Top-1 disagreements

In [14]:
display(semantic_artifacts['baseline_comparison'])
display(semantic_artifacts['disagreements'][[
    'feature_name', 'description', 'expected_canonical_feature',
    'baseline_top_candidate', 'challenger_top_candidate',
    'challenger_top1_cosine', 'challenger_second_candidate',
    'challenger_top2_cosine', 'challenger_score_gap',
    'disagreement_outcome',
]])

,matcher,top1_accuracy,top3_accuracy,exact_accuracy,no_match_accuracy,inverse_accuracy
0,baseline_rapidfuzz_tfidf,0.96875,1.00000,1.0,0.363636,0.666667
1,challenger_sentence_embedding,0.84375,0.96875,1.0,0.000000,0.666667


,feature_name,description,expected_canonical_feature,baseline_top_candidate,challenger_top_candidate,challenger_top1_cosine,challenger_second_candidate,challenger_top2_cosine,challenger_score_gap,disagreement_outcome
11,CNT_DLQ_12M,Number of delinquency events during the previo...,delinquency_frequency,delinquency_frequency,delinquency_recency,0.627897,delinquency_frequency,0.588544,0.039353,baseline_correct_embedding_wrong
14,DTI_CURR,Current total monthly debt obligations divided...,debt_service_burden,income_capacity,debt_yield,0.529932,income_capacity,0.512276,0.017656,both_wrong_different
15,PTI_ORIG,Original contractual loan payment divided by b...,debt_service_burden,debt_service_burden,income_capacity,0.477216,loan_to_cost,0.464705,0.012511,baseline_correct_embedding_wrong
26,CAPACITY_TREND,Year-over-year growth in household income,income_growth,income_growth,real_disposable_income_growth,0.580844,income_growth,0.575527,0.005317,baseline_correct_embedding_wrong
28,COV_METRIC_B,Operating earnings divided by contractual inte...,interest_coverage,interest_coverage,profitability_margin,0.476689,interest_coverage,0.453712,0.022977,baseline_correct_embedding_wrong
41,NUM_MISSED_PMTS_24M,Count of missed or delinquent payment events i...,delinquency_frequency,delinquency_frequency,delinquency_severity,0.420983,delinquency_frequency,0.419315,0.001668,baseline_correct_embedding_wrong
44,NOI_DSCR,Property NOI divided by annual principal and i...,debt_service_coverage,debt_service_coverage,debt_yield,0.695402,debt_service_coverage,0.598513,0.096889,baseline_correct_embedding_wrong
46,TOTAL_DEBT_ASSETS,Total interest-bearing debt divided by total a...,financial_leverage,financial_leverage,debt_yield,0.475523,debt_service_burden,0.432712,0.042811,baseline_correct_embedding_wrong
58,CLTV,Combined first and second lien balances divide...,loan_to_value,loan_to_value,debt_yield,0.406778,loan_to_value,0.384575,0.022203,baseline_correct_embedding_wrong
60,RR_METRIC,Recovery rate on previously defaulted exposures,,cash_flow_growth,delinquency_severity,0.324909,prior_adverse_event,0.313122,0.011787,both_wrong_different


## Previous failures and delinquency-concept checks

In [15]:
display(semantic_artifacts['focus_features'][[
    'feature_name', 'expected_canonical_feature', 'baseline_top_candidate',
    'challenger_top_candidate', 'challenger_top1_cosine',
    'challenger_second_candidate', 'challenger_top2_cosine',
    'embedding_only_top_candidate', 'embedding_only_top1_cosine',
]])

,feature_name,expected_canonical_feature,baseline_top_candidate,challenger_top_candidate,challenger_top1_cosine,challenger_second_candidate,challenger_top2_cosine,embedding_only_top_candidate,embedding_only_top1_cosine
10,MAX_DPD_24M,delinquency_severity,delinquency_severity,delinquency_severity,0.357412,delinquency_recency,0.336479,delinquency_severity,0.357412
11,CNT_DLQ_12M,delinquency_frequency,delinquency_frequency,delinquency_recency,0.627897,delinquency_frequency,0.588544,delinquency_recency,0.627897
12,MTH_SINCE_DLQ,delinquency_recency,delinquency_recency,delinquency_recency,NaN,NaN,NaN,delinquency_recency,0.644773
14,DTI_CURR,debt_service_burden,income_capacity,debt_yield,0.529932,income_capacity,0.512276,debt_yield,0.529932
47,MONTHLY_DEBT_INC,debt_service_burden,income_capacity,income_capacity,0.597617,debt_service_burden,0.571828,income_capacity,0.597617


## Embedding score distributions and expected no-match cases

In [16]:
display(semantic_artifacts['score_distributions'])
display(semantic_artifacts['expected_no_match_cases'][[
    'feature_name', 'description', 'challenger_top_candidate',
    'challenger_top1_cosine', 'challenger_second_candidate',
    'challenger_top2_cosine', 'challenger_score_gap',
]])

,group,metric,count,minimum,p25,median,mean,p75,maximum
0,correct_embedding_top1,embedding_only_top1_cosine,53,0.357412,0.533901,0.623513,0.613390,0.682883,0.819394
1,correct_embedding_top1,embedding_only_score_gap,53,0.002472,0.094883,0.142804,0.160297,0.203686,0.389876
2,incorrect_embedding_top1,embedding_only_top1_cosine,11,0.406778,0.476106,0.522948,0.528348,0.589230,0.695402
3,incorrect_embedding_top1,embedding_only_score_gap,11,0.001668,0.015083,0.022203,0.027923,0.032571,0.096889
4,expected_no_match,embedding_only_top1_cosine,11,0.104591,0.210838,0.248924,0.268781,0.331129,0.548416
5,expected_no_match,embedding_only_score_gap,11,0.004747,0.010112,0.022923,0.032348,0.048260,0.093255


,feature_name,description,challenger_top_candidate,challenger_top1_cosine,challenger_second_candidate,challenger_top2_cosine,challenger_score_gap
60,RR_METRIC,Recovery rate on previously defaulted exposures,delinquency_severity,0.324909,prior_adverse_event,0.313122,0.011787
62,BORROWER_ZIP_CODE,Borrower's postal or ZIP code,debt_yield,0.263474,loan_to_cost,0.237793,0.025681
63,CUSTOMER_ID,Unique customer identifier,collections,0.104591,fixed_charge_coverage,0.081668,0.022923
64,PROPERTY_TYPE,"Property category such as office, retail, indu...",property_occupancy,0.337349,property_value_growth,0.309391,0.027958
65,STATE_CODE,State or geographic region code,debt_yield,0.188308,credit_quality_score,0.116063,0.072245
66,LOAN_PURPOSE,"Purpose of borrowing such as purchase, refinan...",loan_to_cost,0.548416,loan_to_value,0.455161,0.093255
67,INDUSTRY_CODE,Industry classification code for the obligor,debt_yield,0.236687,profitability_margin,0.168124,0.068563
68,BRANCH_ID,Identifier of the originating or servicing branch,loan_to_value,0.233369,cash_flow_to_debt,0.219851,0.013518
69,CURRENCY_CODE,Currency denomination of the facility,loan_to_value,0.248924,utilization,0.244177,0.004747
70,ORIGINATION_CHANNEL,Channel through which the loan was originated,loan_to_cost,0.350181,loan_to_value,0.343462,0.006719


## Model-independent semantic adjudication contract

This example exercises the strict Python boundary only. It does not call an LLM and does not change matcher behavior.

In [ ]:
from functions.semantic_adjudication_contract import (
    AdjudicationInput, AdjudicationOutput, Candidate, InputFeature,
    determine_review_required, validate_adjudication_result,
)

In [ ]:
dti_adjudication_input = AdjudicationInput(
    input_feature=InputFeature(
        name='DTI_CURR', description='Current debt-to-income burden'
    ),
    candidates=[
        Candidate(
            canonical_feature='income_capacity',
            definition='Capacity to generate income available for debt payment.',
            representations=['income capacity'],
        ),
        Candidate(
            canonical_feature='debt_service_burden',
            definition='Debt payment obligation relative to available income.',
            representations=['debt-to-income ratio', 'DTI'],
        ),
        Candidate(
            canonical_feature='financial_leverage',
            definition='Use of debt relative to capital or assets.',
            representations=['financial leverage'],
        ),
    ],
)

valid_match = AdjudicationOutput(
    decision='MATCH',
    selected_candidate='debt_service_burden',
    representation_orientation='SAME',
    reason='DTI measures debt-service burden relative to income.',
)
validated_match = validate_adjudication_result(dti_adjudication_input, valid_match)
validated_match.model_dump(mode='json')

In [ ]:
invented_candidate_output = AdjudicationOutput(
    decision='MATCH',
    selected_candidate='debt_to_income_ratio',
    representation_orientation='SAME',
    reason='This candidate was invented and was not supplied.',
)

try:
    validate_adjudication_result(dti_adjudication_input, invented_candidate_output)
except ValueError as exc:
    print(f'Expected validation failure: {exc}')

In [ ]:
not_directional = AdjudicationOutput(
    decision='NOT_DIRECTIONAL',
    selected_candidate=None,
    representation_orientation=None,
    reason='PROPERTY_TYPE is categorical rather than intrinsically increasing or decreasing.',
)
insufficient_context = AdjudicationOutput(
    decision='INSUFFICIENT_CONTEXT',
    selected_candidate=None,
    representation_orientation=None,
    reason='X_VAR_017 has no description from which to infer economic meaning.',
)

{
    'not_directional': not_directional.model_dump(mode='json'),
    'insufficient_context': insufficient_context.model_dump(mode='json'),
    'exact_match_review_required': determine_review_required('deterministic_exact', False),
    'semantic_adjudication_review_required': determine_review_required('semantic_candidate', True),
}